# Kapruka Gift-Concierge Agent Test
This notebook demonstrates the full **Agent Orchestrator** pipeline, including Memory Layer (ST/LT), Query Routing, and the Reflection Loop.

In [8]:
import os
import sys
from loguru import logger
from dotenv import load_dotenv

# Add project root and src to sys.path to prevent ModuleNotFoundError
PROJECT_ROOT = os.path.abspath('..')
SRC_ROOT     = os.path.join(PROJECT_ROOT, 'src')

if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)
if SRC_ROOT not in sys.path: sys.path.append(SRC_ROOT)

# Load environment variables (.env should be in project root)
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

from infrastructure.llm.llm_provider import (
    get_chat_llm, 
    get_router_llm, 
    get_extractor_llm
)
from services.ingest_service.embeddings import get_default_embeddings
from memory.memory_ops import MemoryDistiller
from agents.tools.logsitic_agent import LogisticAlert
from agents.tools.rag_tool import RAGTool
from agents.orchestrator import AgentOrchestrator

print("✅ Libraries and paths configured!")

✅ Libraries and paths configured!


### 1. Initialize Components

In [9]:
llm_chat      = get_chat_llm(temperature=0)
llm_router    = get_router_llm(temperature=0)
llm_extractor = get_extractor_llm(temperature=0)
embedder      = get_default_embeddings()

# Memory distiller with a clean test profile
test_profile_path = os.path.join(PROJECT_ROOT, "data", "notebook_test_profiles.json")
memory = MemoryDistiller(llm=llm_extractor, profile_path=test_profile_path)

logistic_tool = LogisticAlert(llm=llm_extractor)
rag_tool      = RAGTool(embedder=embedder, llm=llm_chat)

agent = AgentOrchestrator(
    llm_chat=llm_chat,
    llm_router=llm_router,
    memory=memory,
    logistic_tool=logistic_tool,
    rag_tool=rag_tool,
)

print("✅ Orchestrator and tools ready!")

2026-03-29 12:44:35.947 | DEBUG    | memory.memory_ops:__init__:21 - [MemoryDistiller] Initialized with profile path: e:\my career life\AI Enginert Essential\All Mini Projects\Mini project 03\data\notebook_test_profiles.json
2026-03-29 12:44:44.505 | INFO     | infrastructure.db.qdrant_client:get_qdrant_client:106 - Connected to Qdrant Cloud at https://6af70757-4a58-4027-b104-4753e2ef73ce.us-east4-0.gcp.cloud.qdrant.io
2026-03-29 12:44:46.145 | INFO     | services.chat_service.cag_cache:__init__:80 - ✓ CAG cache ready (Qdrant collection='cag_cache', dim=1536, threshold=0.90)
2026-03-29 12:44:46.148 | INFO     | agents.tools.rag_tool:__init__:87 - RAGTool initialised: CAG cache (CAGCache(collection='cag_cache', threshold=0.9, ttl=86400s, entries=2, backend='qdrant')) -> CRAG (k=4, expanded_k=8, threshold=0.60)


✅ Orchestrator and tools ready!


### 2. Test Case: Logistics Query
Check if the router correctly identifies the `logistic` route and the tool provides Sri Lankan district delivery zones.

In [10]:
response = agent.chat("Can you deliver a gift to Galle?")

print(f"Route: {response.route}")
print(f"Answer: {response.answer}")

2026-03-29 12:44:46.590 | INFO     | memory.memory_ops:recaller:51 - [MemoryDistiller] Recalling context for: 'Can you deliver a gift to Galle?'
2026-03-29 12:44:46.592 | DEBUG    | memory.lt_store:get_profiles:68 - [LongTermMemory] Profile file empty or missing — returning []
2026-03-29 12:44:46.595 | DEBUG    | memory.lt_store:get_profiles:68 - [LongTermMemory] Profile file empty or missing — returning []
2026-03-29 12:44:46.597 | INFO     | agents.router:route:83 - [QueryRouter] Routing: 'Can you deliver a gift to Galle?'
2026-03-29 12:44:47.306 | DEBUG    | agents.router:route:109 - [QueryRouter] Tokens — in:310 out:51 total:361
2026-03-29 12:44:47.309 | INFO     | agents.router:route:125 - [QueryRouter] → route='logistic' confidence=0.80 model='llama-3.1-8b-instant' reason='User is asking about delivery to a specific location'
2026-03-29 12:44:47.312 | INFO     | agents.orchestrator:_dispatch:180 - [Orchestrator] → LogisticAlert: 'Galle'
2026-03-29 12:44:47.314 | INFO     | agents

Route: logistic
Answer: Yes, we can deliver a gift to Galle. The delivery is feasible and typically takes about 2-3 days. If you need assistance with selecting a gift or placing an order, feel free to ask!


### 3. Test Case: Short-Term + Long-Term Memory (Intent-based)
Tell the agent to *remember* a preference, then ask about it in the next turn to see if the extraction worked.

In [11]:
print("--- Turn 1: Save ---")
agent.chat("Please remember that my brother hates anything with cinnamon but loves fruit cakes.")

print("\n--- Turn 2: Recall ---")
response = agent.chat("What should I get for my brother?")

print(f"Answer: {response.answer}")

2026-03-29 12:44:49.399 | INFO     | memory.memory_ops:recaller:51 - [MemoryDistiller] Recalling context for: 'Please remember that my brother hates anything with cinnamon but loves fruit cakes.'
2026-03-29 12:44:49.402 | DEBUG    | memory.lt_store:get_profiles:68 - [LongTermMemory] Profile file empty or missing — returning []
2026-03-29 12:44:49.403 | DEBUG    | memory.lt_store:get_profiles:68 - [LongTermMemory] Profile file empty or missing — returning []
2026-03-29 12:44:49.406 | INFO     | agents.router:route:83 - [QueryRouter] Routing: 'Please remember that my brother hates anything with cinnamon but loves fruit cak'


--- Turn 1: Save ---


2026-03-29 12:44:49.802 | DEBUG    | agents.router:route:109 - [QueryRouter] Tokens — in:382 out:57 total:439
2026-03-29 12:44:49.806 | INFO     | agents.router:route:125 - [QueryRouter] → route='direct' confidence=0.80 model='llama-3.1-8b-instant' reason='User is providing additional context for a previous conversation'
2026-03-29 12:44:51.092 | DEBUG    | agents.orchestrator:_synthesise:233 - [Orchestrator] Draft tokens — in:143 out:32 total:175
2026-03-29 12:44:51.093 | DEBUG    | agents.orchestrator:chat:120 - [Orchestrator] No profiles in LT memory — skipping reflection.
2026-03-29 12:44:51.094 | DEBUG    | memory.St_store:add_turn:97 - [ShortTermMemory] Added 'user' turn. Buffer size: 3
2026-03-29 12:44:51.096 | DEBUG    | memory.St_store:add_turn:97 - [ShortTermMemory] Added 'assistant' turn. Buffer size: 4
2026-03-29 12:44:51.097 | INFO     | memory.memory_ops:saving_memory:86 - [MemoryDistiller] Short-term buffer updated (user + assistant turns).
2026-03-29 12:44:51.098 | INFO


--- Turn 2: Recall ---


2026-03-29 12:44:51.853 | DEBUG    | agents.router:route:109 - [QueryRouter] Tokens — in:456 out:54 total:510
2026-03-29 12:44:51.854 | INFO     | agents.router:route:125 - [QueryRouter] → route='rag' confidence=0.80 model='llama-3.1-8b-instant' reason='User is asking for gift recommendations based on brother's preferences'
2026-03-29 12:44:51.857 | INFO     | agents.orchestrator:_dispatch:190 - [Orchestrator] → RAGTool: 'fruit cake without cinnamon'
2026-03-29 12:45:05.307 | DEBUG    | services.chat_service.cag_cache:set:219 - CAG cache SET: 'fruit cake without cinnamon' → point=f3c25a2b-e52d-4146-8c23-1fe8a2965140
2026-03-29 12:45:05.309 | INFO     | services.chat_service.cag_service:generate:68 - CAG cache MISS -> cached for: fruit cake without cinnamon
2026-03-29 12:45:07.808 | DEBUG    | agents.orchestrator:_synthesise:233 - [Orchestrator] Draft tokens — in:356 out:95 total:451
2026-03-29 12:45:07.810 | INFO     | agents.orchestrator:_reflect:252 - [Orchestrator] Reflecting draft 

Answer: I recommend the **Fruity Garden Spectacle Vanilla Sponge Cake** for your brother. It's a delicious fruit-based cake that doesn't contain cinnamon, making it a perfect fit for his preferences. The price is RS. 6,800. You can check it out [here](https://www.kapruka.com/buyonline/fruity-garden-spectacle-vanill/kid/cake00ka001860). Let me know if you need any more help!


### 4. Test Case: Reflection Loop (Safety Gate)
If we explicitly ask for something that violates the previously saved preference, the reflection loop should catch and revise it.

In [12]:
request = "Can you suggest a nice cinnamon-spiced apple cake for my brother?"
response = agent.chat(request)

print(f"Violated: {response.violated}")
print(f"Answer: {response.answer}")

2026-03-29 12:45:09.144 | INFO     | memory.memory_ops:recaller:51 - [MemoryDistiller] Recalling context for: 'Can you suggest a nice cinnamon-spiced apple cake for my brother?'
2026-03-29 12:45:09.148 | DEBUG    | memory.lt_store:get_profiles:72 - [LongTermMemory] Loaded 1 profile(s) from e:\my career life\AI Enginert Essential\All Mini Projects\Mini project 03\data\notebook_test_profiles.json
2026-03-29 12:45:09.152 | DEBUG    | memory.lt_store:get_profiles:72 - [LongTermMemory] Loaded 1 profile(s) from e:\my career life\AI Enginert Essential\All Mini Projects\Mini project 03\data\notebook_test_profiles.json
2026-03-29 12:45:09.155 | INFO     | agents.router:route:83 - [QueryRouter] Routing: 'Can you suggest a nice cinnamon-spiced apple cake for my brother?'
2026-03-29 12:45:09.916 | DEBUG    | agents.router:route:109 - [QueryRouter] Tokens — in:574 out:64 total:638
2026-03-29 12:45:09.919 | INFO     | agents.router:route:125 - [QueryRouter] → route='rag' confidence=0.80 model='llama

Violated: False
Answer: Since your brother dislikes cinnamon, I wouldn't recommend a cinnamon-spiced apple cake for him. Instead, I suggest sticking with fruit cakes that he loves. The **Fruity Garden Spectacle Vanilla Sponge Cake** is a great option, as it’s fruit-based and doesn't contain cinnamon. If you need more suggestions or help, just let me know!


### 5. Final Answer Generation (Direct Route)
This section demonstrates how the agent generates a professional and helpful final response for general gift-related questions, even when no specific tool (RAG/Logistic) is needed.

In [13]:
print("--- Scenario: Professional Gift Advice ---")
response = agent.chat("What are the most popular gift categories on Kapruka for Mother's Day?")

print(f"Route: {response.route}")
print(f"Final Answer:\n{response.answer}")

2026-03-29 12:45:21.161 | INFO     | memory.memory_ops:recaller:51 - [MemoryDistiller] Recalling context for: 'What are the most popular gift categories on Kapruka for Mother's Day?'
2026-03-29 12:45:21.164 | DEBUG    | memory.lt_store:get_profiles:72 - [LongTermMemory] Loaded 1 profile(s) from e:\my career life\AI Enginert Essential\All Mini Projects\Mini project 03\data\notebook_test_profiles.json
2026-03-29 12:45:21.169 | DEBUG    | memory.lt_store:get_profiles:72 - [LongTermMemory] Loaded 1 profile(s) from e:\my career life\AI Enginert Essential\All Mini Projects\Mini project 03\data\notebook_test_profiles.json
2026-03-29 12:45:21.171 | INFO     | agents.router:route:83 - [QueryRouter] Routing: 'What are the most popular gift categories on Kapruka for Mother's Day?'


--- Scenario: Professional Gift Advice ---


2026-03-29 12:45:21.928 | DEBUG    | agents.router:route:109 - [QueryRouter] Tokens — in:668 out:50 total:718
2026-03-29 12:45:21.931 | INFO     | agents.router:route:125 - [QueryRouter] → route='rag' confidence=1.00 model='llama-3.1-8b-instant' reason='User is asking for gift recommendations'
2026-03-29 12:45:21.934 | INFO     | agents.orchestrator:_dispatch:190 - [Orchestrator] → RAGTool: 'Mother's Day gift categories'
2026-03-29 12:45:29.908 | DEBUG    | services.chat_service.cag_cache:set:219 - CAG cache SET: 'Mother's Day gift categories' → point=67f6b73b-d9d0-498c-8033-de442a6c614e
2026-03-29 12:45:29.927 | INFO     | services.chat_service.cag_service:generate:68 - CAG cache MISS -> cached for: Mother's Day gift categories
2026-03-29 12:45:32.540 | DEBUG    | agents.orchestrator:_synthesise:233 - [Orchestrator] Draft tokens — in:567 out:121 total:688
2026-03-29 12:45:32.546 | INFO     | agents.orchestrator:_reflect:252 - [Orchestrator] Reflecting draft against 1 profile(s)...
202

Route: rag
Final Answer:
Here’s a revised gift recommendation for Mother's Day that ensures a safe and delightful selection:

Some of the most popular gift categories on Kapruka for Mother's Day include:

1. **Personalized Gifts**: Items like customized mugs, photo frames, and jewelry are always cherished.
2. **Flowers**: Fresh flower arrangements are a classic choice that brighten any occasion.
3. **Cakes**: Consider special occasion cakes, especially fruit cakes, which are a favorite!
4. **Spa and Wellness Products**: Pampering items like bath sets and skincare products make for a relaxing gift.

If you need specific recommendations or help with a gift, feel free to ask!


### Cleanup (Optional)

In [14]:
if os.path.exists(test_profile_path):
    os.remove(test_profile_path)
    print(f"🗑 Cleaned up {test_profile_path}")

🗑 Cleaned up e:\my career life\AI Enginert Essential\All Mini Projects\Mini project 03\data\notebook_test_profiles.json
